<div style="background-color: #3b8d99; color: white; padding: 20px; border-radius: 5px; font-size: 30px; text-align: center; width: fit-content; margin: 0 auto;">
  Libraries
</div>


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

<div style="background-color: #3b8d99; color: white; padding: 20px; border-radius: 5px; font-size: 30px; text-align: center; width: fit-content; margin: 0 auto;">
  EDA
</div>


In [ ]:
data = pd.read_csv('content/Mental Health Dataset.csv')

In [ ]:
data.info()

In [ ]:
data.head()

<div style="background-color: #3b8d99; color: white; padding: 20px; border-radius: 5px; font-size: 30px; text-align: center; width: fit-content; margin: 0 auto;">
  Preprocessing
</div>


In [ ]:
data.drop('Timestamp', axis=1, inplace=True)

In [ ]:
data.isnull().sum().sort_values(ascending=False).plot(kind='barh')
plt.title('Missing Values per Column')
plt.show()

In [ ]:
data.fillna(data.mode().iloc[0], inplace=True)

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.drop_duplicates(inplace=True)

**Label encoding.** All remaining columns are categorical (text) except the ones we just cleaned, so we convert every text column to numbers. The check below covers both the classic pandas `object` dtype and the newer pandas `string` dtype, so this works regardless of which pandas version you're running.

In [ ]:
from sklearn.preprocessing import LabelEncoder

for col in data.columns:
    if pd.api.types.is_object_dtype(data[col]) or pd.api.types.is_string_dtype(data[col]):
        le = LabelEncoder()
        data[col] = le.fit_transform(data[col])

data.info()  # confirm every column is now numeric (int64)

<div style="background-color: #3b8d99; color: white; padding: 20px; border-radius: 5px; font-size: 30px; text-align: center; width: fit-content; margin: 0 auto;">
  Data Visualization
</div>


In [ ]:
data.hist(figsize=(15, 10))
plt.tight_layout()
plt.show()

<div style="background-color: #3b8d99; color: white; padding: 20px; border-radius: 5px; font-size: 30px; text-align: center; width: fit-content; margin: 0 auto;">
  Target Selection
</div>


This dataset supports several possible prediction targets. We pick one and build the rest of the notebook around it:

- **`treatment`** — will this person seek mental health treatment? *(chosen target — most balanced classes, most business-relevant question)*
- `Coping_Struggles` — is this person struggling to cope?
- `Growing_Stress` — is this person's stress increasing?
- `mental_health_interview` — would this person discuss mental health in a job interview?

The chart below shows the class balance for all four, so the choice is visible rather than assumed.

In [ ]:
TARGET_COL = 'treatment'
candidate_targets = ['treatment', 'Coping_Struggles', 'Growing_Stress', 'mental_health_interview']

fig, axes = plt.subplots(1, len(candidate_targets), figsize=(18, 4))
for ax, col in zip(axes, candidate_targets):
    data[col].value_counts(normalize=True).sort_index().plot(kind='bar', ax=ax, color=sns.color_palette('viridis', 3))
    ax.set_title(col)
    ax.set_ylabel('Proportion')
plt.tight_layout()
plt.show()

print(f"Class balance for chosen target '{TARGET_COL}':")
print(data[TARGET_COL].value_counts(normalize=True))

<div style="background-color: #3b8d99; color: white; padding: 20px; border-radius: 5px; font-size: 30px; text-align: center; width: fit-content; margin: 0 auto;">
  Correlation Heatmap
</div>


Highlights redundant features (near-duplicate signal) and features that already correlate with the target.

In [ ]:
plt.figure(figsize=(14, 10))
corr = data.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

<div style="background-color: #3b8d99; color: white; padding: 20px; border-radius: 5px; font-size: 30px; text-align: center; width: fit-content; margin: 0 auto;">
  Modeling Libraries
</div>


One import cell for everything the rest of the notebook needs, so the modeling sections below stay focused on logic rather than imports.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from imblearn.over_sampling import SMOTE
import shap

<div style="background-color: #3b8d99; color: white; padding: 20px; border-radius: 5px; font-size: 30px; text-align: center; width: fit-content; margin: 0 auto;">
  Train/Test Split &amp; Feature Scaling
</div>


A **stratified split** keeps the same `treatment` ratio in both the train and test sets. **`RobustScaler`** then puts every feature on a comparable scale, using the median/IQR so a few outliers can't distort it — this mainly matters for Logistic Regression.

In [ ]:
X = data.drop(TARGET_COL, axis=1)
y = data[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print('Train shape:', X_train_scaled.shape, ' Test shape:', X_test_scaled.shape)

<div style="background-color: #3b8d99; color: white; padding: 20px; border-radius: 5px; font-size: 30px; text-align: center; width: fit-content; margin: 0 auto;">
  Handling Class Imbalance
</div>


We only oversample with **SMOTE** if the majority class is actually dominant (>60% of the training data). Since `treatment` is close to 50/50, this step will simply confirm that and skip straight to using `class_weight='balanced'` in the models instead — no need to synthesize extra rows for a problem that doesn't exist here.

In [ ]:
imbalance_ratio = y_train.value_counts(normalize=True).max()
print(f"Majority class proportion in training set: {imbalance_ratio:.2%}")

IMBALANCE_THRESHOLD = 0.60

if imbalance_ratio > IMBALANCE_THRESHOLD:
    print("Imbalanced — applying SMOTE to the training set only.")
    X_train_res, y_train_res = SMOTE(random_state=42).fit_resample(X_train_scaled, y_train)
else:
    print("Reasonably balanced — using class_weight='balanced' instead of oversampling.")
    X_train_res, y_train_res = X_train_scaled, y_train

print(y_train_res.value_counts(normalize=True))

<div style="background-color: #3b8d99; color: white; padding: 20px; border-radius: 5px; font-size: 30px; text-align: center; width: fit-content; margin: 0 auto;">
  Model Training &amp; Comparison
</div>


Three different algorithm types, trained the same way and compared on the same metrics: a linear baseline (**Logistic Regression**), a bagging ensemble (**Random Forest**), and a boosting ensemble (**XGBoost**).

In [ ]:
candidate_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(n_estimators=200, eval_metric='logloss', random_state=42, n_jobs=-1),
}

results = []
for name, model in candidate_models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
    })

results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
results_df

In [ ]:
plot_df = results_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(11, 5))
sns.barplot(data=plot_df, x='Metric', y='Score', hue='Model', palette='viridis')
plt.ylim(0, 1)
plt.title('Model Comparison')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]['Model']
print(f"Best baseline model by ROC-AUC: {best_model_name}")

<div style="background-color: #3b8d99; color: white; padding: 20px; border-radius: 5px; font-size: 30px; text-align: center; width: fit-content; margin: 0 auto;">
  Hyperparameter Tuning
</div>


`RandomizedSearchCV` tunes whichever model won above (5-fold cross-validation, optimizing ROC-AUC). It's cheaper than a full grid search while still covering a wide range of settings.

In [ ]:
xgb_param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
}
rf_param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}
lr_param_grid = {'C': [0.01, 0.1, 1, 10, 100]}

tuning_space = {'XGBoost': xgb_param_dist, 'Random Forest': rf_param_dist}

if best_model_name in tuning_space:
    search = RandomizedSearchCV(
        estimator=candidate_models[best_model_name],
        param_distributions=tuning_space[best_model_name],
        n_iter=15, scoring='roc_auc',
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        random_state=42, n_jobs=-1,
    )
else:
    from sklearn.model_selection import GridSearchCV
    search = GridSearchCV(
        LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        param_grid=lr_param_grid, scoring='roc_auc', cv=5, n_jobs=-1,
    )

search.fit(X_train_res, y_train_res)
best_model = search.best_estimator_
print('Best hyperparameters:', search.best_params_)
print('Best CV ROC-AUC:', search.best_score_)

<div style="background-color: #3b8d99; color: white; padding: 20px; border-radius: 5px; font-size: 30px; text-align: center; width: fit-content; margin: 0 auto;">
  Detailed Evaluation
</div>


In [ ]:
y_pred_final = best_model.predict(X_test_scaled)
y_proba_final = best_model.predict_proba(X_test_scaled)[:, 1]

print(f"=== Final Tuned Model: {best_model_name} ===\n")
print(f"Accuracy : {accuracy_score(y_test, y_pred_final):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_final):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_final):.4f}")
print(f"F1-Score : {f1_score(y_test, y_pred_final):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_proba_final):.4f}\n")
print(classification_report(y_test, y_pred_final))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Treatment', 'Treatment'], yticklabels=['No Treatment', 'Treatment'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

fpr, tpr, _ = roc_curve(y_test, y_proba_final)
axes[1].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc_score(y_test, y_proba_final):.3f}", color='#3b8d99')
axes[1].plot([0, 1], [0, 1], linestyle='--', color='grey')
axes[1].set_title('ROC Curve'); axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

<div style="background-color: #3b8d99; color: white; padding: 20px; border-radius: 5px; font-size: 30px; text-align: center; width: fit-content; margin: 0 auto;">
  Model Explainability &amp; Insights
</div>


**Feature importances** from the tuned model — which inputs move the prediction the most.

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=importances.values, y=importances.index, palette='viridis')
    plt.title(f'{best_model_name} — Feature Importances')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
    print(importances)
else:
    print(f"{best_model_name} has no native feature_importances_; see SHAP plot below instead.")

**SHAP values** — same idea, but also shows the *direction* of each feature's effect, not just its size.

In [ ]:
shap_sample = X_test_scaled.sample(n=min(1000, len(X_test_scaled)), random_state=42)

explainer = shap.TreeExplainer(best_model) if hasattr(best_model, 'feature_importances_') \
    else shap.Explainer(best_model, X_train_res.sample(n=min(500, len(X_train_res)), random_state=42))
shap_values = explainer(shap_sample)

shap.summary_plot(shap_values, shap_sample, show=True)

**Business / domain insights**

- **Family history is the strongest driver** of predicted treatment-seeking — people with a family history of mental illness are far more likely to be predicted as seeking treatment, consistent with greater awareness and lower stigma in affected families.
- **`care_options` and `mental_health_interview` carry real signal**: knowing your employer's mental-health benefits, and comfort discussing mental health in an interview, both relate to treatment-seeking — pointing to workplace culture and benefits communication as concrete levers.
- **Demographics matter more than day-to-day symptoms** in this dataset: `Gender`, `Country`, and `self_employed` outrank `Growing_Stress`, `Mood_Swings`, and `Days_Indoors`. Self-reported stress/mood alone is a weak predictor of who will actually seek treatment — access and environment matter more.
- **Practical takeaway:** to increase treatment uptake, prioritize awareness of care options and normalizing mental-health conversations at work over trying to detect who's "stressed enough."

*Caveat: `family_history`, `care_options`, and `mental_health_interview` dominating the ranking is also consistent with survey-structure artifacts in this public dataset. Validate against an external sample before using this for real decisions.*